### CRISP-DM Phase 4.3 - Modeling : Visualization dashboard

In [ ]:
import pandas as pd
from iso3166 import countries_by_alpha3
from pycountry_convert import country_alpha3_to_country_alpha2
from pycountry_convert import country_alpha2_to_continent_code

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# Load the datasets
legislative_coverage_country = pd.read_csv('data/legislative_coverage_country.csv')
hazard_intensity_country = pd.read_csv('data/hazard_intensity_country.csv')
correlation_country = pd.read_csv('data/correlation_country.csv')

# Drop country and continent names to recompute them from ISO codes, ensures consistency
for df in [legislative_coverage_country, hazard_intensity_country, correlation_country]:
    df.drop(columns=[c for c in ['Country_name', 'Continent_name'] if c in df.columns], inplace=True)

In [ ]:
VARIABLES = ['2m_temperature', 'Instantaneous_wind_gust', 'Sea_level_anomaly', 
             'Snowmelt', 'SPEI', 'Total_precipitation']

hazard_variable_dict = {'flood': 'Total_precipitation', 'drought': 'SPEI', 
                        'temperature_extremes': '2m_temperature', 'sea_level_rise': 'Sea_level_anomaly', 
                        'storm': 'Instantaneous_wind_gust', 'melting': 'Snowmelt'}

hazards = list(hazard_variable_dict.keys())

Country dataframe

In [ ]:
def iso_to_name(iso):
    try:
        result = countries_by_alpha3.get(iso)
        return result.name if result else None
    except:
        return None

country_fix_dict = {
    'Bolivia, Plurinational State of': 'Bolivia',
    'Venezuela, Bolivarian Republic of': 'Venezuela',
    'Iran, Islamic Republic of': 'Iran',
    "Côte d'Ivoire": 'Ivory Coast',
    'Congo, Democratic Republic of the': 'Democratic Republic of the Congo',
    'Congo': 'Republic of the Congo',
    'Tanzania, United Republic of': 'Tanzania',
    'Türkiye': 'Turkey',
    'Moldova, Republic of': 'Moldova',
    'Taiwan, Province of China': 'Taiwan',
    'Palestine, State of': 'Palestine'
}

continent_fix_dict = {'TL': 'AS', 'AQ': 'AQ', 'TF': 'AQ'}

def country_to_continent(country):
    if country == 'XKX':
        return 'EU'
    try:
        country = country_alpha3_to_country_alpha2(country)
        if country in continent_fix_dict:
            return continent_fix_dict[country]
        else:
            continent = country_alpha2_to_continent_code(country)
            return continent
    except:
        return None

In [ ]:
# Merge legislative coverage and hazard intensity
country = legislative_coverage_country.merge(hazard_intensity_country[['Country', 'Year'] + [v for v in VARIABLES]], on=['Country', 'Year'], how='inner')
country.rename(columns={'Count_national': 'Coverage'}, inplace=True)

# Keep intensity values only for the relevant hazard variable
country['Intensity'] = country.apply(
    lambda row: row[hazard_variable_dict.get(row['Hazard'], '')] 
    if hazard_variable_dict.get(row['Hazard']) else None, axis=1)
country.drop(columns=VARIABLES, inplace=True)

# Merge with correlation data
country = country.merge(correlation_country, on=['Country', 'Hazard'], how='left')

# Add country names
country['Country_name'] = country['Country'].apply(iso_to_name)
country['Country_name'] = country['Country_name'].replace(country_fix_dict)

# Add continent
country['Continent'] = country['Country'].apply(country_to_continent)
country['Continent_name'] = country['Continent'].map({'AF': 'Africa', 'AS': 'Asia', 'EU': 'Europe',
                                                      'NA': 'North America', 'OC': 'Oceania', 
                                                      'SA': 'South America', 'AQ': 'Antarctica'})

# Add hazard name
country['Hazard_name'] = country['Hazard'].str.replace('_', ' ').str.title()

In [ ]:
# Country to continent mapping
world = gpd.read_file("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip")
iso_fixes = {'France': 'FRA', 'Norway': 'NOR', 'South Sudan': 'SSD', 'Kosovo': 'XKX',
             'Northern Cyprus': 'CYN', 'Somaliland': 'SOM'}
for country_name, correct_iso in iso_fixes.items():
    world.loc[world['NAME'] == country_name, 'ISO_A3'] = correct_iso

world['Continent'] = world['ISO_A3'].apply(country_to_continent)

continent_colors = {'AF': 'orange', 'AS': 'blue', 'EU': 'green', 
                    'NA': 'yellow', 'OC': 'purple', 'SA': 'red'}
continent_names = {'AF': 'Africa', 'AS': 'Asia', 'EU': 'Europe',
                    'NA': 'North America', 'OC': 'Oceania', 'SA': 'South America'}

world['color'] = world['Continent'].map(continent_colors)

fig, ax = plt.subplots(figsize=(15, 8))
world[world['color'].notna()].plot(color=world.loc[world['color'].notna(), 'color'], ax=ax,
                                     edgecolor='white', linewidth=0.3)
world[world['color'].isna()].plot(color='lightgrey', ax=ax, edgecolor='white', linewidth=0.3)
ax.axis('off')
patches = [mpatches.Patch(color=color, label=continent_names[code]) for code, color in continent_colors.items()]
ax.legend(handles=patches, loc='center left', ncol=2, frameon=False, fontsize=11)

plt.tight_layout()
plt.savefig('outputs/continent_mapping.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
## Export .csv
dashboard_data = country[['Country', 'Country_name', 'Continent', 'Continent_name', 'Year', 'Hazard', 
                          'Hazard_name', 'Coverage', 'Intensity', 'Rho', 'p_value']]
dashboard_data = dashboard_data[dashboard_data['Hazard'].isin(hazards)].copy()
dashboard_data.to_csv('outputs/dashboard_data.csv', index=False)